# 하이퍼파라미터 튜닝
[mini_gpt_mps.ipynb](mini_gpt_mps.ipynb)의 스케일업 모델(튜닝 전)은 값을 임의로 정한 것이었음. 같은 데이터·장비에서 체계적으로 튜닝하고, 최적값으로 학습한 결과를 튜닝 전 모델과 같은 시간 기준으로 비교함.

### 튜닝 방법
1. 목표 지표는 val loss임 (학습에 쓰지 않은 구간의 다음 글자 예측 오차). train loss는 쓰지 않음.
2. 같은 시간(compute) 예산으로 비교함. 큰 모델은 스텝당 느려서 같은 스텝 수로 비교하면 불공정함.
3. 한 번에 하나씩 좁힘: 학습률, 모델 크기 순서. 학습률이 성능에 가장 민감함.
4. 탐색은 짧게, 최적값으로만 길게 학습함.
5. 훈련 기법 개선: 워밍업 + 코사인 감쇠, gradient clipping, AdamW(β₂=0.95, weight decay 0.1)
6. 모든 실험을 같은 고정 val 배치로 평가함.

한계: 짧은 탐색(학습률 300스텝, 크기 120초)에서 고른 값은 긴 훈련의 최적값과 다를 수 있음. 특히 학습률은 짧은 탐색에서 높게 편향되는 경향이 있음.

### 0. 준비 (mini_gpt_mps.ipynb와 동일한 데이터·모델)

In [1]:
import time, math, html
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import HTML, display

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(42)
print("사용 장치:", device, "| torch", torch.__version__)

def sync(dev):
    if dev == "mps":
        torch.mps.synchronize()

사용 장치: mps | torch 2.14.0


In [2]:
text = open("data/input.txt", encoding="utf-8").read().strip()

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
split = int(len(data) * 0.95)
train_data, val_data = data[:split], data[split:]
print(f"글자 수: {len(text):,} | 어휘 크기: {vocab_size:,} | train {len(train_data):,} / val {len(val_data):,}")

글자 수: 29,953,814 | 어휘 크기: 1,473 | train 28,456,123 / val 1,497,691


In [3]:
def get_batch(context_length, batch_size, dev, split_name="train"):
    d = train_data if split_name == "train" else val_data
    ix = torch.randint(0, len(d) - context_length - 1, (batch_size,))
    offs = torch.arange(context_length)
    x = d[ix[:, None] + offs]
    y = d[ix[:, None] + offs + 1]
    return x.to(dev), y.to(dev)

In [4]:
class SelfAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # Q·K, 인과 마스크, softmax, ·V 한 번에
        out = out.transpose(1, 2).reshape(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = SelfAttention(embed_dim, n_heads)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Linear(embed_dim * 4, embed_dim),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, context_length, n_heads, n_layers):
        super().__init__()
        self.context_length = context_length
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.blocks = nn.Sequential(*[Block(embed_dim, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, targets=None):
        B, T = x.shape
        h = self.token_emb(x) + self.pos_emb(torch.arange(T, device=x.device))
        h = self.ln_f(self.blocks(h))
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, start_text, max_new_tokens=50):
        self.eval()
        dev = next(self.parameters()).device
        ids = torch.tensor([encode(start_text)], dtype=torch.long, device=dev)
        for _ in range(max_new_tokens):
            logits, _ = self(ids[:, -self.context_length:])
            probs = F.softmax(logits[:, -1, :], dim=-1)
            ids = torch.cat([ids, torch.multinomial(probs, num_samples=1)], dim=1)
        self.train()
        return decode(ids[0].tolist())

### 1. 튜닝용 훈련 함수
워밍업 후 코사인 감쇠 학습률 스케줄, gradient clipping(1.0), AdamW(β=(0.9, 0.95), weight decay 0.1)을 지원함.
평가는 처음에 뽑아 둔 고정 val 배치 40개(컨텍스트 256, 배치 64)로 해서 실험 간 val loss를 직접 비교할 수 있음.

In [5]:
CTX, BS = 256, 64

g = torch.Generator().manual_seed(0)
val_batches = []
for _ in range(40):
    ix = torch.randint(0, len(val_data) - CTX - 1, (BS,), generator=g)
    offs = torch.arange(CTX)
    val_batches.append((val_data[ix[:, None] + offs], val_data[ix[:, None] + offs + 1]))

def make_model(c):
    return MiniGPT(vocab_size, c["embed_dim"], CTX, c["n_heads"], c["n_layers"]).to(device)

@torch.no_grad()
def eval_fixed(model, amp=True):
    model.eval()
    ls = []
    for x, y in val_batches:
        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=amp):
            ls.append(model(x.to(device), y.to(device))[1].item())
    model.train()
    return sum(ls) / len(ls)

def lr_at(step, total, lr, warmup=0.05, min_ratio=0.1):
    w = max(1, int(total * warmup))
    if step < w:
        return lr * (step + 1) / w
    p = (step - w) / max(1, total - w)
    return lr * (min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * p)))

def run(c, steps, improved=True, eval_points=0, seed=42, verbose=False):
    """c: embed_dim, n_heads, n_layers, lr. improved=False면 튜닝 전 설정과 같은 방식(스케줄·clip 없음)."""
    torch.manual_seed(seed)
    model = make_model(c)
    if improved:
        opt = torch.optim.AdamW(model.parameters(), lr=c["lr"], betas=(0.9, 0.95), weight_decay=0.1)
    else:
        opt = torch.optim.AdamW(model.parameters(), lr=c["lr"])
    curve = []
    marks = {int(steps * (k + 1) / eval_points) for k in range(eval_points)} if eval_points else set()
    sync(device); t0 = time.perf_counter()
    for step in range(steps):
        if improved:
            for gp in opt.param_groups:
                gp["lr"] = lr_at(step, steps, c["lr"])
        x, y = get_batch(CTX, BS, device)
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            _, loss = model(x, y)
        opt.zero_grad(); loss.backward()
        if improved:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        if (step + 1) in marks:
            sync(device)
            v = eval_fixed(model)
            curve.append((step + 1, v))
            if verbose:
                print(f"  step {step+1}/{steps}: val {v:.3f} ({time.perf_counter()-t0:.0f}초)")
            sync(device); 
    sync(device)
    elapsed = time.perf_counter() - t0
    val = eval_fixed(model)
    if not curve or curve[-1][0] != steps:
        curve.append((steps, val))
    params = sum(p.numel() for p in model.parameters())
    return dict(model=model, val=val, time=elapsed, steps=steps, params=params, curve=curve, cfg=c)

def step_time(c, n=5):
    """스텝당 시간(초) 추정: 워밍업 2스텝 제외 n스텝 평균"""
    m = make_model(c); opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    def one():
        x, y = get_batch(CTX, BS, device)
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            _, loss = m(x, y)
        opt.zero_grad(); loss.backward(); opt.step()
    for _ in range(2): one()
    sync(device); t = time.perf_counter()
    for _ in range(n): one()
    sync(device)
    return (time.perf_counter() - t) / n

### 2. 1단계: 학습률 스윕
스케일업 구조(d=384, 6층)를 고정하고 학습률만 바꿔 300스텝씩 학습함. 학습률은 로그 간격으로 잡음.
val loss가 가장 낮은 값을 후보로 함. 너무 크면 발산하고 너무 작으면 300스텝 안에 학습이 부족함.

In [6]:
base_arch = dict(embed_dim=384, n_heads=6, n_layers=6)
lr_grid = [3e-4, 6e-4, 1e-3, 2e-3, 4e-3]
lr_results = {}
for lr in lr_grid:
    r = run({**base_arch, "lr": lr}, steps=300)
    lr_results[lr] = r["val"]
    print(f"lr {lr:.0e}: val loss {r['val']:.3f}  ({r['time']:.0f}초)")
best_lr = min(lr_results, key=lambda k: (lr_results[k] if lr_results[k] == lr_results[k] else 1e9))
print(f"\n최적 학습률 후보: {best_lr:.0e} (val {lr_results[best_lr]:.3f})")

lr 3e-04: val loss 3.701  (112초)


lr 6e-04: val loss 3.557  (112초)


lr 1e-03: val loss 3.431  (112초)


lr 2e-03: val loss 3.265  (112초)


lr 4e-03: val loss 3.316  (112초)

최적 학습률 후보: 2e-03 (val 3.265)


### 3. 2단계: 모델 크기, 같은 시간(120초)으로 비교
학습률은 1단계 값으로 고정하고 모델 크기 후보를 각각 120초 동안 학습함(스텝당 시간을 먼저 재서 스텝 수를 정함).
작은 모델을 많이 학습하는 쪽과 큰 모델을 적게 학습하는 쪽 중 어느 쪽이 유리한지 보는 단계임. 헤드당 차원은 모두 64로 유지함. 후보 중 가장 작은 모델이 d=192이며, 더 작은 모델은 시험하지 않음.

In [7]:
size_grid = [
    dict(embed_dim=192, n_heads=3, n_layers=4),
    dict(embed_dim=256, n_heads=4, n_layers=6),
    dict(embed_dim=384, n_heads=6, n_layers=6),
    dict(embed_dim=512, n_heads=8, n_layers=8),
]
BUDGET = 120
size_results = []
for arch in size_grid:
    c = {**arch, "lr": best_lr}
    st = step_time(c)
    steps_ = max(30, int(BUDGET / st))
    r = run(c, steps=steps_)
    size_results.append(r)
    print(f"d={arch['embed_dim']:>3} L={arch['n_layers']} ({r['params']/1e6:5.1f}M) | {steps_:>4}스텝 | val {r['val']:.3f} | {r['time']:.0f}초")
best_size = min(size_results, key=lambda r: r["val"])["cfg"]
print("\n같은 시간 기준 최적 구조:", {k: v for k, v in best_size.items() if k != "lr"})

d=192 L=4 (  2.4M) |  958스텝 | val 2.972 | 123초


d=256 L=6 (  5.6M) |  517스텝 | val 3.088 | 123초


d=384 L=6 ( 11.9M) |  333스텝 | val 3.218 | 124초


d=512 L=8 ( 26.9M) |  180스텝 | val 3.527 | 125초

같은 시간 기준 최적 구조: {'embed_dim': 192, 'n_heads': 3, 'n_layers': 4}


### 4. 기준선: 튜닝 전 모델을 같은 방식으로 재측정
튜닝 전 설정(d=384, 6층, lr 3e-4, 스케줄·clip 없음, 2000스텝)을 이 노트북의 고정 val 배치로 다시 평가함.
걸린 시간이 최종 훈련의 시간 예산이 됨.

In [8]:
baseline = run({**base_arch, "lr": 3e-4}, steps=2000, improved=False)
print(f"튜닝 전: val {baseline['val']:.3f} | {baseline['time']:.0f}초 | {baseline['params']:,} 파라미터")
T_BUDGET = baseline["time"]

튜닝 전: val 2.934 | 721초 | 11,878,593 파라미터


### 5. 최종: 최적값으로 같은 시간 동안 학습
2단계에서 고른 구조, 1단계에서 고른 학습률, 개선한 훈련 기법으로 기준선과 같은 시간(초)만큼 학습함.
중간 val loss를 10구간으로 기록해 학습 곡선을 확인함.

In [9]:
st = step_time(best_size)
final_steps = int(T_BUDGET / st)
print(f"구조 {({k: v for k, v in best_size.items() if k != 'lr'})}, lr {best_size['lr']:.0e}, {final_steps}스텝 예정 (예산 {T_BUDGET:.0f}초)")
final = run(best_size, steps=final_steps, eval_points=10, verbose=True)
print(f"\n최종: val {final['val']:.3f} | {final['time']:.0f}초 | {final['params']:,} 파라미터")

구조 {'embed_dim': 192, 'n_heads': 3, 'n_layers': 4}, lr 2e-03, 5799스텝 예정 (예산 721초)


  step 579/5799: val 3.141 (75초)


  step 1159/5799: val 2.925 (149초)


  step 1739/5799: val 3.033 (224초)


  step 2319/5799: val 3.097 (299초)


  step 2899/5799: val 3.065 (373초)


... (2 outputs left out)

  step 4639/5799: val 2.968 (598초)


  step 5219/5799: val 2.915 (672초)


  step 5799/5799: val 2.927 (747초)



최종: val 2.927 | 747초 | 2,396,097 파라미터


### 6. 튜닝 전후 비교
같은 데이터, 같은 고정 val 배치, 같은 시간 예산에서 비교함. bits/char = loss ÷ ln2, perplexity = e^loss임.
튜닝 후 설정은 구조, 학습률, 훈련 기법이 함께 바뀐 것이라 각 요인의 효과는 분리되지 않음.

In [10]:
prompt = "오늘은 "
torch.manual_seed(1)
gen_base = baseline["model"].generate(prompt, max_new_tokens=100)
torch.manual_seed(1)
gen_final = final["model"].generate(prompt, max_new_tokens=100)

def desc(r):
    c = r["cfg"]
    return f"d={c['embed_dim']}, {c['n_layers']}층, {c['n_heads']}헤드, lr {c['lr']:.0e}"
cell = lambda s: html.escape(s).replace("\n", "<br>")
tds = lambda f: "".join(f"<td style='padding:6px 10px;border:1px solid #888'>{f(r)}</td>" for r in (baseline, final))
row = lambda label, f: f"<tr><td style='padding:6px 10px;border:1px solid #888'><b>{label}</b></td>{tds(f)}</tr>"

display(HTML(f"""
<table style="font-size:15px;border-collapse:collapse;width:100%">
<tr><th style="padding:6px 10px;border:1px solid #888"></th>
<th style="padding:6px 10px;border:1px solid #888">튜닝 전</th>
<th style="padding:6px 10px;border:1px solid #888">튜닝 후</th></tr>
{row("설정", desc)}
{row("훈련 기법", lambda r: "기본 AdamW" if r is baseline else "워밍업+코사인, clip, wd 0.1, β₂=0.95")}
{row("파라미터", lambda r: f"{r['params']:,}")}
{row("스텝 수", lambda r: f"{r['steps']:,}")}
{row("훈련 시간", lambda r: f"{r['time']:.0f}초")}
{row("val loss", lambda r: f"{r['val']:.3f}")}
{row("perplexity", lambda r: f"{math.exp(r['val']):.1f}")}
{row("bits/char", lambda r: f"{r['val']/math.log(2):.2f}")}
{row("생성 샘플", lambda r: cell(gen_base if r is baseline else gen_final))}
</table>
"""))
gain = (baseline["val"] - final["val"]) / baseline["val"] * 100
print(f"val loss {baseline['val']:.3f} → {final['val']:.3f} ({gain:.1f}% 감소) | perplexity {math.exp(baseline['val']):.1f} → {math.exp(final['val']):.1f}")

,튜닝 전,튜닝 후
설정,"d=384, 6층, 6헤드, lr 3e-04","d=192, 4층, 3헤드, lr 2e-03"
훈련 기법,기본 AdamW,"워밍업+코사인, clip, wd 0.1, β₂=0.95"
파라미터,"11,878,593","2,396,097"
스텝 수,"2,000","5,799"
훈련 시간,721초,747초
val loss,2.934,2.927
perplexity,18.8,18.7
bits/char,4.23,4.22
생성 샘플,오늘은 롤과의 방향을 생각했습니다. 주요 꽃은 유린티키를 대비하여 취급을 받을 수 있습니다. 출발자비중째가 앞에 있어 세계에 올해끼 나입회적으로 긴장대 모든 것과 ‘너우레를 인도하하지 않,오늘은 롤과의 방에 적합한 수얻을 수 있는데요. 인티도 사랑하는 견고한 제품에 사랑하는 사냥한다고 말씀하신다면 수많은 내년에 올해 예측할 줄 아는 그대로 죽일의 목표로 느껴지는 것이 목을


val loss 2.934 → 2.927 (0.2% 감소) | perplexity 18.8 → 18.7
